# Train Mamba on Z24

Notebook này chạy `train_mamba.py` với **within-recording split 4/1/1**:

- Input đầy đủ: `(27, 10000)`, không crop/downsample.
- Với mọi `(class, setup)`: window 0–3 train, window 4 validation, window 5 test.
- Cả 17 class và 9 setup xuất hiện trong mọi tập; các tập dùng cửa sổ khác nhau của cùng bản ghi.
- Augmentation chỉ áp dụng cho train.
- Checkpoint tốt nhất được chọn bằng validation loss; test không tham gia chọn model.
- t-SNE dùng cùng tập test ở sau epoch 1 và sau epoch cuối để so sánh embedding.

Artifacts được lưu trong `results/<run_name>/`. Mamba ở repo là selective SSM đúng recurrence nhưng dùng Python scan; một lần train đầy đủ có thể chậm trên Windows.

In [ ]:
import json
from pathlib import Path

import torch
from IPython.display import Image, display

from train_mamba import TrainConfig, train

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Cấu hình

Để kiểm tra nhanh pipeline trước khi train đầy đủ, đặt `epochs=1`, `max_train_batches=2`, `skip_tsne=True`. Cấu hình dưới đây là cấu hình train thật.

In [ ]:
config = TrainConfig(
    run_name="mamba_within_recording_noaug_seed42",
    epochs=100,
    patience=12,
    batch_size=8,
    learning_rate=3e-4,
    weight_decay=1e-4,
    hidden=64,
    layers=2,
    stem_stride=25,
    seed=42,
    num_workers=0,  # đổi thành 4 khi chạy trên Kaggle
    use_augmentation=False,
    split_mode="within_recording",
    max_train_batches=0,
    skip_tsne=False,
)
config

## 2. Huấn luyện

Cell này tạo `trainlog.txt`, cập nhật `mamba_final.pt` mỗi epoch, lưu `mamba_best.pt` khi validation loss tốt hơn, và sinh t-SNE sau epoch 1 và epoch cuối.

In [ ]:
run_dir, metrics = train(config)
print("Completed:", run_dir)

## 3. Metrics

Các metrics gồm accuracy, balanced accuracy, precision/recall/F1 macro và weighted, ROC-AUC, top-3 accuracy, Cohen's kappa, MCC và kết quả từng lớp.

In [ ]:
saved_metrics = json.loads((Path(run_dir) / "metrics.json").read_text(encoding="utf-8"))
print("Best epoch:", saved_metrics["best_epoch"])
print("Final epoch:", saved_metrics["final_epoch"])
print("\nTEST METRICS")
for name, value in saved_metrics["test"].items():
    if name != "per_class":
        print(f"{name:24s}: {value}")

## 4. Training curves, confusion matrix và ROC

In [ ]:
for filename in ["training_curves.png", "confusion_matrix.png", "accuracy_per_class.png",
                 "accuracy_per_class_by_setup.png", "roc_curve.png"]:
    print(filename)
    display(Image(filename=str(Path(run_dir) / filename)))

## 5. t-SNE epoch 1 và epoch cuối

Mỗi thời điểm có t-SNE 2D tô màu theo damage scenario/setup, t-SNE 3D và file tọa độ/embedding `.npz`.

In [ ]:
for folder, title in [("tsne_epoch_001", "Epoch 1"), ("tsne_epoch_final", "Final epoch")]:
    print(title)
    display(Image(filename=str(Path(run_dir) / folder / "tsne_2d.png")))
    display(Image(filename=str(Path(run_dir) / folder / "tsne_3d.png")))

## 6. Danh sách đầu ra

Các file bắt buộc gồm `trainlog.txt`, `mamba_best.pt`, `mamba_final.pt`, `metrics.json`, `history.json`, predictions và hai thư mục t-SNE.

In [ ]:
for path in sorted(Path(run_dir).rglob("*")):
    if path.is_file():
        print(path.relative_to(run_dir), f"({path.stat().st_size / 1024:.1f} KiB)")